# 📖 Lab 3: Politeness — robots.txt + Domain Rate Limiting

**Non-functional requirement:** *Respect robots.txt and don't overload servers.*

An impolite crawler is the classic way to get your IP banned. Politeness has two halves:

1. **robots.txt** — which paths am I allowed to fetch, and how fast?
2. **Per-domain rate limiting** — max ~1 request/sec *per domain*, enforced across *all* crawler workers.

The second half is the hard one, because "all workers" means the counter can't live in
one process's memory. It has to be shared state — Redis — and the check has to be atomic.

## Learning Objectives

- Measure what an impolite crawler actually does to a single origin server
- Enforce 1 req/sec/domain across concurrent workers with an atomic Redis lock
- Honour `Disallow` and `Crawl-delay` from a real robots.txt
- See *why* jitter matters, with numbers instead of assertions
- Learn where Python's stdlib robots.txt parser is wrong


## 🛠️ Setup

```bash
cd 06-system-designs/web-crawler
docker compose up -d
```

Select the **"Python 3 (.venv)"** kernel.

In [ ]:
import http.server
import random
import socket
import socketserver
import threading
import time
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

import redis
import requests

redis_client = redis.Redis(host="localhost", port=6382, decode_responses=True)
redis_client.flushdb()
print(f"✅ Redis: {redis_client.ping()}")

## 🖥️ A Local Origin Server We Can Measure

Politeness is about what the *server* experiences, so we need a server that tells us.

We run a tiny threaded HTTP server on localhost that timestamps **every** request it
receives. That gives us the one number that matters: **peak requests per second seen by
this origin**. Crawling real websites to prove a point about overloading them would be
rather impolite.

It serves:

| Path | Content |
|------|---------|
| `/robots.txt` | `Disallow: /private/` + `Crawl-delay: 1` |
| `/page/<n>` | a small HTML page |
| `/private/secret` | a page our crawler must never fetch |

In [ ]:
# ── Request log: every hit the origin server receives, with a timestamp ──
hits: list[tuple[str, float]] = []
hits_lock = threading.Lock()

ROBOTS_TXT = """User-agent: *
Disallow: /private/
Crawl-delay: 1
"""


class OriginHandler(http.server.BaseHTTPRequestHandler):
    """A polite-crawler test target that records exactly when it was hit."""

    def do_GET(self):
        with hits_lock:
            hits.append((self.path, time.perf_counter()))

        if self.path == "/robots.txt":
            body, ctype = ROBOTS_TXT.encode(), "text/plain"
        else:
            body = f"<html><body><h1>{self.path}</h1><p>content</p></body></html>".encode()
            ctype = "text/html"

        self.send_response(200)
        self.send_header("Content-Type", ctype)
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def log_message(self, *args):  # keep the notebook output clean
        pass


class ThreadedHTTPServer(socketserver.ThreadingMixIn, http.server.HTTPServer):
    daemon_threads = True


# Bind port 0 so the OS hands us a free port — no clashes with other labs.
_probe = socket.socket()
_probe.bind(("127.0.0.1", 0))
PORT = _probe.getsockname()[1]
_probe.close()

origin_server = ThreadedHTTPServer(("127.0.0.1", PORT), OriginHandler)
threading.Thread(target=origin_server.serve_forever, daemon=True).start()

ORIGIN = f"http://127.0.0.1:{PORT}"
print(f"✅ Origin server listening on {ORIGIN}")
print(f"   robots.txt:\n{requests.get(ORIGIN + '/robots.txt', timeout=5).text}")

### Measuring the origin's pain

`peak_rps()` slides a 1-second window over the origin's request log and returns the
largest number of requests that landed inside any one window. Our politeness budget is
**1 req/sec/domain**, so this number *is* the score.

In [ ]:
def reset_hits():
    with hits_lock:
        hits.clear()


def origin_report(label: str, exclude_robots: bool = True) -> int:
    """Peak requests/sec observed by the origin server, plus a mini timeline."""
    with hits_lock:
        log = [(p, t) for p, t in hits if not (exclude_robots and p == "/robots.txt")]

    if not log:
        print(f"  {label}: the origin received 0 requests.")
        return 0

    times = sorted(t for _, t in log)
    start = times[0]
    peak = max(sum(1 for u in times if t <= u < t + 1.0) for t in times)

    print(f"  {label}")
    print(f"    requests received : {len(times)}")
    print(f"    wall clock        : {times[-1] - start:.2f}s")
    print(f"    peak requests/sec : {peak}   (politeness budget is 1)")
    # One '#' per request, bucketed into 500 ms columns.
    span = max(times[-1] - start, 0.001)
    buckets = [0] * (int(span / 0.5) + 1)
    for t in times:
        buckets[int((t - start) / 0.5)] += 1
    print(f"    timeline (0.5s per column): |{'|'.join('#' * b if b else '.' for b in buckets)}|")
    return peak


print("✅ Measurement helpers ready.")

## 💥 The Impolite Crawler

This is the crawler almost everybody writes first: a thread pool that pulls URLs off the
frontier and fetches them as fast as the network allows. It has no idea that 8 of those
URLs belong to the same poor origin server.

In [ ]:
TARGET_URLS = [f"{ORIGIN}/page/{i}" for i in range(8)]


def impolite_fetch(url: str):
    requests.get(url, timeout=5, headers={"User-Agent": "EducationalCrawlerBot/1.0"})


reset_hits()
print("💥 8 workers, 8 URLs, all on one host, no politeness:\n")

threads = [threading.Thread(target=impolite_fetch, args=(u,)) for u in TARGET_URLS]
for t in threads:
    t.start()
for t in threads:
    t.join()

impolite_peak = origin_report("IMPOLITE")
print(f"\n  ⚠️  The origin absorbed {impolite_peak}x its politeness budget in a single second.")
print("     On a real site this is what earns you a 429, then a firewall rule.")

## 🔒 The Fix: One Atomic Domain Lock in Redis

The workers are separate threads here, and separate *machines* in production, so the
"when did we last hit this domain?" state has to be shared. Redis `SET key NX EX <delay>`
is the whole mechanism:

- `NX` — only set if the key does not exist. This is the atomic part: exactly one caller
  wins, even with hundreds of workers racing.
- `EX <delay>` — the key evaporates after `delay` seconds, which *is* the rate limit.

A worker that loses the race doesn't fetch. It sleeps a little and tries again.

> **Trade-off:** this lock costs one Redis round-trip per attempt, and a worker blocked on
> a busy domain is a worker doing nothing. Real crawlers avoid the idle time by keeping a
> *per-domain* frontier and handing each worker a URL from a domain that is currently free,
> rather than blocking on whatever URL happened to come off a global queue. The lock is the
> safety net, not the scheduler.

In [ ]:
DEFAULT_CRAWL_DELAY = 1.0  # seconds between requests to the same domain

robots_cache: dict[str, RobotFileParser] = {}
USER_AGENT = "EducationalCrawlerBot/1.0 (system-design-labs)"


def get_robots(base: str) -> RobotFileParser:
    """Fetch + cache robots.txt for an origin. One fetch per domain, ever."""
    if base in robots_cache:
        return robots_cache[base]

    rp = RobotFileParser()
    try:
        resp = requests.get(f"{base}/robots.txt", timeout=5,
                            headers={"User-Agent": USER_AGENT})
        if resp.status_code == 200:
            rp.parse(resp.text.splitlines())
        elif resp.status_code in (401, 403):
            rp.parse(["User-agent: *", "Disallow: /"])   # RFC 9309: treat as full block
        else:
            rp.parse(["User-agent: *", "Allow: /"])      # 404 → no rules → crawl away
    except requests.RequestException:
        rp.parse(["User-agent: *", "Allow: /"])

    robots_cache[base] = rp
    return rp


def acquire_domain_lock(domain: str, crawl_delay: float) -> bool:
    """Atomic 'may I hit this domain right now?'. Redis SET NX EX is the entire trick."""
    # Redis EX takes whole seconds; PX lets us honour sub-second delays too.
    return bool(redis_client.set(
        f"domain_lock:{domain}", "1", nx=True, px=int(crawl_delay * 1000)
    ))


def polite_fetch(url: str, max_wait: float = 15.0) -> dict:
    """robots.txt check → domain lock → fetch. Returns why it did or didn't fetch."""
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"
    rp = get_robots(base)

    # 1. Is this path even allowed? If not we never touch the network.
    if not rp.can_fetch(USER_AGENT, url):
        return {"url": url, "status": "disallowed_by_robots", "fetched": False}

    # 2. robots.txt may dictate a slower rate than our default.
    delay = rp.crawl_delay(USER_AGENT) or rp.crawl_delay("*") or DEFAULT_CRAWL_DELAY

    # 3. Wait for our turn on this domain.
    deadline = time.perf_counter() + max_wait
    waits = 0
    while not acquire_domain_lock(parsed.netloc, float(delay)):
        if time.perf_counter() > deadline:
            return {"url": url, "status": "gave_up_waiting", "fetched": False}
        waits += 1
        # Jitter here is deliberate — see the thundering-herd section below.
        time.sleep(0.05 + random.uniform(0, 0.05))

    resp = requests.get(url, timeout=5, headers={"User-Agent": USER_AGENT})
    return {"url": url, "status": "success", "fetched": True,
            "size": len(resp.text), "lock_waits": waits}


print("✅ Polite fetcher ready.")

In [ ]:
reset_hits()
redis_client.flushdb()
robots_cache.clear()

print("🔒 Same 8 workers, same 8 URLs — now behind the domain lock:\n")

polite_results: list[dict] = []
results_lock = threading.Lock()


def polite_worker(url: str):
    r = polite_fetch(url)
    with results_lock:
        polite_results.append(r)


threads = [threading.Thread(target=polite_worker, args=(u,)) for u in TARGET_URLS]
for t in threads:
    t.start()
for t in threads:
    t.join()

polite_peak = origin_report("POLITE")

fetched = sum(1 for r in polite_results if r["fetched"])
total_waits = sum(r.get("lock_waits", 0) for r in polite_results)
print(f"\n  fetched {fetched}/{len(TARGET_URLS)} pages, "
      f"{total_waits} lock waits across all workers")
print(f"\n  📉 peak req/sec: {impolite_peak} (impolite) → {polite_peak} (polite)")
print("     Same work, same workers. The origin now sees a steady trickle instead of a spike.")
print(f"     Cost: the crawl took ~{len(TARGET_URLS)}s instead of ~0s. Politeness is *slow* —")
print("     which is exactly why you crawl thousands of domains in parallel, not one fast.")

## 🤖 robots.txt: `Disallow` Is Checked Before the Network

Our origin server disallows `/private/`. The proof that the crawler respects it isn't a
log line saying "skipped" — it's the origin server confirming it never received the
request at all.

In [ ]:
reset_hits()
redis_client.flushdb()

probe_urls = [
    f"{ORIGIN}/page/100",        # allowed
    f"{ORIGIN}/private/secret",  # Disallow: /private/
    f"{ORIGIN}/private/keys",    # Disallow: /private/
]

print("🤖 robots.txt enforcement:\n")
for url in probe_urls:
    r = polite_fetch(url)
    icon = "✅" if r["fetched"] else "⛔"
    print(f"  {icon} {url:<45} {r['status']}")

with hits_lock:
    private_hits = [p for p, _ in hits if p.startswith("/private")]
print(f"\n  Origin server saw {len(private_hits)} requests to /private/  ← must be 0")
assert not private_hits, "robots.txt Disallow was not honoured!"

rp = get_robots(ORIGIN)
print(f"  Crawl-delay advertised by the origin: {rp.crawl_delay(USER_AGENT)}s")
print("  robots.txt itself was fetched once and cached — not once per URL.")

### ⚠️ Where the stdlib parser lies to you

Python's `urllib.robotparser` is convenient but **not** RFC 9309 compliant. It normalises
each rule path through `urlunparse`, which **drops the query string**. So Google's

```
Disallow: /?
```

(meaning "don't crawl the homepage *with query parameters*") is stored as

```
Disallow: /
```

...which blocks the entire site. Run the cell below and watch it claim that
`https://www.google.com/` is off limits. It isn't.

Production crawlers use a spec-compliant matcher — Google's own open-sourced
[`robotstxt`](https://github.com/google/robotstxt), or the `protego` package in Python —
which also support the `*` and `$` wildcards that `robotparser` handles only partially.

> This cell talks to the live internet. If you are offline it degrades to a message
> instead of failing.

In [ ]:
def stdlib_says(url: str) -> str:
    parsed = urlparse(url)
    rp = get_robots(f"{parsed.scheme}://{parsed.netloc}")
    return "allowed" if rp.can_fetch(USER_AGENT, url) else "DISALLOWED"


try:
    requests.get("https://www.google.com/robots.txt", timeout=8,
                 headers={"User-Agent": USER_AGENT}).raise_for_status()
except Exception as exc:  # offline / blocked — skip, don't fail the lab
    print(f"⏭️  Skipping live check ({type(exc).__name__}).")
else:
    print("🐛 What urllib.robotparser thinks of google.com:\n")
    for url in ["https://www.google.com/",
                "https://www.google.com/search?q=test",
                "https://www.google.com/maps/"]:
        print(f"  {url:<45} {stdlib_says(url)}")

    google = get_robots("https://www.google.com")
    entry = google.default_entry or google.entries[0]
    dropped = [ln for ln in entry.rulelines if ln.path == "/"]
    print(f"\n  Rule lines that ended up as bare '/': {[str(ln) for ln in dropped]}")
    print("  Google's file says 'Disallow: /?' — the '?' was stripped, so it reads as")
    print("  'Disallow: /' and the whole domain looks blocked. The parser is wrong,")
    print("  not Google. Use protego or google/robotstxt in production.")

## 🎲 Jitter: Measuring the Thundering Herd

Every blocked worker is sitting on a timer waiting for the same lock to expire. Without
jitter they all wake at the *same instant*, all hammer Redis in the same millisecond, one
wins and the rest go back to sleep — still in lockstep. The herd never disperses.

Jitter randomises each sleep so the retries fan out.

We measure two things over an identical workload (6 workers, one domain, run until every
worker gets through):

| Metric | Meaning |
|--------|---------|
| **peak simultaneous retries** | most retry attempts landing inside the same 25 ms window |
| **total Redis round-trips** | wasted work — every failed `SET NX` is a network call |

Note what *doesn't* change: wall-clock time. The 1 req/sec/domain budget sets that, and
jitter cannot beat physics. Jitter buys you a calmer Redis, not a faster crawl.

In [ ]:
HERD_WORKERS = 6
RETRY_PERIOD = 0.25   # base sleep between lock attempts
BUCKET = 0.025        # 25 ms — "at the same time" for our purposes


def herd_trial(use_jitter: bool) -> dict:
    """All workers race for one domain lock; report how clustered their retries were."""
    redis_client.delete("domain_lock:herd.example")
    retry_times: list[float] = []
    attempts = 0
    bookkeeping = threading.Lock()
    # A barrier makes every worker start in the same instant — worst case on purpose.
    gate = threading.Barrier(HERD_WORKERS)

    def worker():
        nonlocal attempts
        gate.wait()
        attempt_no = 0
        while True:
            now = time.perf_counter()
            got = redis_client.set("domain_lock:herd.example", "1", nx=True, px=1000)
            with bookkeeping:
                attempts += 1
                if attempt_no > 0:      # the first burst is unavoidable; retries are the story
                    retry_times.append(now)
            if got:
                return
            attempt_no += 1
            time.sleep(RETRY_PERIOD + (random.uniform(0, RETRY_PERIOD) if use_jitter else 0))

    started = time.perf_counter()
    threads = [threading.Thread(target=worker) for _ in range(HERD_WORKERS)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    elapsed = time.perf_counter() - started

    retry_times.sort()
    peak = max((sum(1 for u in retry_times if t <= u < t + BUCKET) for t in retry_times),
               default=0)
    return {"peak_simultaneous": peak, "redis_calls": attempts, "seconds": elapsed}


print(f"🎲 {HERD_WORKERS} workers racing for one domain lock...\n")
no_jitter = herd_trial(use_jitter=False)
with_jitter = herd_trial(use_jitter=True)

print(f"  {'':<28}{'no jitter':>12}{'with jitter':>14}")
print(f"  {'-' * 54}")
for key, label in [("peak_simultaneous", "peak retries in 25ms"),
                   ("redis_calls", "total Redis calls"),
                   ("seconds", "wall clock (s)")]:
    a, b = no_jitter[key], with_jitter[key]
    fmt = (lambda v: f"{v:.1f}") if key == "seconds" else (lambda v: f"{v}")
    print(f"  {label:<28}{fmt(a):>12}{fmt(b):>14}")

saved = 100 * (no_jitter["redis_calls"] - with_jitter["redis_calls"]) / no_jitter["redis_calls"]
print(f"\n  Without jitter the workers stay in lockstep: {no_jitter['peak_simultaneous']} of the")
print(f"  {HERD_WORKERS} retry inside the same 25ms window, over and over.")
print(f"  With jitter the peak drops to {with_jitter['peak_simultaneous']} and Redis handles "
      f"{saved:.0f}% fewer calls.")
print(f"  Wall clock barely moves ({no_jitter['seconds']:.1f}s vs {with_jitter['seconds']:.1f}s) —")
print("  the 1 req/sec/domain budget decides that, not the retry strategy.")

## 🧹 Cleanup

In [ ]:
origin_server.shutdown()
origin_server.server_close()
redis_client.flushdb()
print("✅ Origin server stopped, Redis flushed.")

## ✅ Summary

| Mechanism | Implementation | Purpose | What it costs |
|-----------|---------------|---------|---------------|
| **robots.txt** | fetch once per domain, cache, check before every fetch | respect site policy | one extra request per domain; a stale cache can be wrong for hours |
| **Domain lock** | Redis `SET NX PX <delay>` | exactly 1 req/`delay` per domain, cluster-wide | a Redis round-trip per attempt, and blocked workers idle |
| **Jitter** | random sleep on retry | disperses the herd → fewer Redis calls | none worth mentioning; always do it |
| **Crawl-delay** | overrides the lock TTL | honour site-specific limits | a `Crawl-delay: 30` site becomes almost uncrawlable |

### The numbers we measured

```
impolite : 8 requests to one origin inside one second   (8x over budget)
polite   : peak 1 request/sec, ~8s for the same 8 pages (on budget)
jitter   : peak simultaneous retries 5 → 2, ~30% fewer Redis calls, same wall clock
```

### The honest trade-off

Politeness makes a single domain **slow by design** — 1 page/sec means a 1M-page site
takes 11 days. The throughput in our capacity estimate comes from **breadth**, not depth:
thousands of domains crawled concurrently, each at its own polite trickle. That is also
why the frontier should be partitioned *by domain* rather than being one global FIFO —
a global queue leaves workers blocked on hot domains while cold ones go untouched.

**Next:** Lab 4 — Efficiency (URL + content deduplication, Bloom filters, crawler traps)